In [1]:
!pip install -q datasets

In [2]:
from datasets import load_dataset
import pandas as pd

In [3]:
games_ds = load_dataset("davdsdfd/nba-games","games")
teams_ds = load_dataset("davdsdfd/nba-games","teams")

README.md:   0%|          | 0.00/925 [00:00<?, ?B/s]

games.csv:   0%|          | 0.00/4.50M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29755 [00:00<?, ? examples/s]

teams.csv:   0%|          | 0.00/4.06k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30 [00:00<?, ? examples/s]

In [4]:
games_df = games_ds["train"].to_pandas()
teams_df = teams_ds["train"].to_pandas()

In [5]:
team_names = teams_df[["TEAM_ID", "NICKNAME"]]

games_df = (
    games_df
    .merge(
        team_names,
        left_on="HOME_TEAM_ID",
        right_on="TEAM_ID",
        how="left"
    )
    .rename(columns={"NICKNAME": "HOME_TEAM"})
    .drop(columns="TEAM_ID")
    .merge(
        team_names,
        left_on="VISITOR_TEAM_ID",
        right_on="TEAM_ID",
        how="left"
    )
    .rename(columns={"NICKNAME": "VISITOR_TEAM"})
    .drop(columns="TEAM_ID")
)

In [6]:
games_df["GAME_DATE_EST"] = pd.to_datetime(games_df["GAME_DATE_EST"])
games_df = games_df.sort_values("GAME_DATE_EST").reset_index(drop=True)

base_games = games_df[[
    "GAME_ID",
    "GAME_DATE_EST",
    "SEASON",
    "HOME_TEAM",
    "VISITOR_TEAM",
    "PTS_home",
    "PTS_away",
    "FG_PCT_home",
    "FG_PCT_away",
    "FT_PCT_home",
    "FT_PCT_away",
    "FG3_PCT_home",
    "FG3_PCT_away",
    "AST_home",
    "AST_away",
    "REB_home",
    "REB_away",
    "HOME_TEAM_WINS"
]].copy()

home_games = base_games.copy()
away_games = base_games.copy()

home_games = home_games.rename(columns={
    "HOME_TEAM": "TEAM",
    "VISITOR_TEAM": "OPPONENT",
    "PTS_home": "PTS",
    "PTS_away": "PTS_ALLOWED",
    "FG_PCT_home": "FG_PCT",
    "FT_PCT_home": "FT_PCT",
    "FG3_PCT_home": "FG3_PCT",
    "AST_home": "AST",
    "REB_home": "REB",
    "HOME_TEAM_WINS": "WIN"
})

home_games["IS_HOME"] = 1

away_games = away_games.rename(columns={
    "VISITOR_TEAM": "TEAM",
    "HOME_TEAM": "OPPONENT",
    "PTS_away": "PTS",
    "PTS_home": "PTS_ALLOWED",
    "FG_PCT_away": "FG_PCT",
    "FT_PCT_away": "FT_PCT",
    "FG3_PCT_away": "FG3_PCT",
    "AST_away": "AST",
    "REB_away": "REB"
})

away_games["WIN"] = 1 - away_games["HOME_TEAM_WINS"]
away_games["IS_HOME"] = 0

In [7]:
team_games_df = pd.concat(
    [home_games, away_games],
    ignore_index=True
)

team_games_df = team_games_df.sort_values(
    ["TEAM", "GAME_DATE_EST", "GAME_ID"]
).reset_index(drop=True)

In [8]:
group = team_games_df.groupby(["TEAM", "SEASON"])

team_games_df["GAMES_BEFORE"] = group.cumcount()

team_games_df["WINS_BEFORE"] = group["WIN"].transform(
    lambda x: x.shift(1).cumsum()
)

team_games_df["WIN_PCT_BEFORE"] = (
    team_games_df["WINS_BEFORE"] /
    team_games_df["GAMES_BEFORE"]
)

stats = [
    "PTS",
    "PTS_ALLOWED",
    "FG_PCT",
    "FT_PCT",
    "FG3_PCT",
    "AST",
    "REB"
]

for stat in stats:
    team_games_df[f"{stat}_LAST_10"] = (
        group[stat]
        .transform(
            lambda x: x.shift(1).rolling(10, min_periods=1).mean()
        )
    )

In [9]:
stats = [
    "PTS",
    "PTS_ALLOWED",
    "FG_PCT",
    "FT_PCT",
    "FG3_PCT",
    "AST",
    "REB"
]

for stat in stats:
    team_games_df[f"{stat}_LAST_10"] = (
        team_games_df
        .groupby(["TEAM", "SEASON"])[stat]
        .transform(
            lambda x: x.shift(1).rolling(10, min_periods=1).mean()
        )
    )

In [10]:
team_games_df[
    team_games_df["TEAM"] == "Warriors"
][[
    "GAME_DATE_EST",
    "SEASON",
    "PTS",
    "PTS_LAST_10",
    "WIN",
    "WIN_PCT_BEFORE"
]].tail(15)

,GAME_DATE_EST,SEASON,PTS,PTS_LAST_10,WIN,WIN_PCT_BEFORE
57555,2025-01-15,2024,116.0,105.7,1,0.555556
57556,2025-01-18,2024,122.0,108.1,1,0.565217
57557,2025-01-20,2024,85.0,109.4,0,0.574468
57558,2025-01-22,2024,117.0,108.4,0,0.562500
57559,2025-01-23,2024,131.0,106.2,1,0.551020
57560,2025-01-25,2024,108.0,107.2,0,0.560000
57561,2025-01-28,2024,114.0,108.1,1,0.549020
57562,2025-01-29,2024,116.0,109.7,1,0.557692
57563,2025-01-31,2024,105.0,110.6,0,0.566038
57564,2025-02-03,2024,104.0,111.5,1,0.555556
